# Evaluate Qwen3-4B AVSD LoRA On Math Benchmarks

Standalone vLLM evaluation notebook for GSM8K, AIME24, AIME25, and HMMT25.

Defaults:
- Avg@4
- `max_tokens=38912`
- Qwen3 thinking enabled at inference
- Resume/skip completed examples

In [ ]:
# Mount Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is unavailable; assuming Drive is already mounted or running locally.')

In [ ]:
# Install runtime dependencies with CUDA-aware wheels.
# Restart the runtime/kernel after this cell if vLLM or PyTorch was changed.
import os
import re

# vLLM V1 can fail inside Colab/ipykernel because ipykernel streams do not expose fileno().
os.environ.setdefault('VLLM_USE_V1', '0')
NUMPY_SPEC = 'numpy==2.1.3'


def detect_cuda_version():
    try:
        nvidia_smi_output = '\n'.join(get_ipython().getoutput('nvidia-smi'))
    except Exception:
        return None
    match = re.search(r'CUDA Version:\s*([0-9]+(?:\.[0-9]+)?)', nvidia_smi_output)
    return match.group(1) if match else None


def vllm_torch_backend(cuda_version):
    if not cuda_version:
        raise RuntimeError('No CUDA GPU was detected. vLLM evaluation requires an NVIDIA CUDA runtime.')
    major, minor = (int(part) for part in cuda_version.split('.', 1))
    if major >= 13:
        return 'cu130'
    if major == 12 and minor >= 9:
        return 'cu129'
    if major == 12 and minor >= 8:
        return 'cu128'
    if major == 12 and minor >= 6:
        return 'cu126'
    raise RuntimeError(f'Unsupported CUDA {cuda_version}. Use a runtime with CUDA 12.6+ for vLLM.')


CUDA_VERSION = detect_cuda_version()
TORCH_BACKEND = vllm_torch_backend(CUDA_VERSION)
TORCH_INDEX_URL = f'https://download.pytorch.org/whl/{TORCH_BACKEND}'
print(f'Detected CUDA {CUDA_VERSION}; installing vLLM/PyTorch backend {TORCH_BACKEND}.')
print('Package install logs are intentionally verbose so download/resolve/build progress is visible.')

print('\n=== Install PyTorch CUDA wheels ===')
!pip install -U torch --index-url {TORCH_INDEX_URL} --progress-bar on

print('\n=== Install vLLM ===')
!pip install -U vllm --extra-index-url {TORCH_INDEX_URL} --progress-bar on

print('\n=== Install evaluation dependencies ===')
!pip install -U {NUMPY_SPEC} "transformers>=4.51.0" "datasets>=3.6.0" "peft>=0.14.0" "math-verify>=0.8.0" tqdm pandas pyarrow --progress-bar on

print('\n=== Repair/pin NumPy after resolver changes ===')
!pip install --force-reinstall --no-cache-dir {NUMPY_SPEC} --progress-bar on

import numpy as np
import pandas as pd
import torch
import transformers
import vllm

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('transformers:', transformers.__version__)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'available:', torch.cuda.is_available())
print('vllm:', getattr(vllm, '__version__', 'unknown'))
print('VLLM_USE_V1:', os.environ.get('VLLM_USE_V1'))
print('If imports still fail after this cell, restart the runtime/kernel and run from the first cell again.')

In [ ]:
# Paths.
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_Project')
BENCHMARK_DIR = DRIVE_ROOT / 'benchmarks'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'qwen3_4b_avsd' / 'checkpoint-500'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'qwen3_4b_avsd_avg4'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = 'Qwen/Qwen3-4B'
BENCHMARKS = ['gsm8k', 'aime24', 'aime25', 'hmmt25']

print('Benchmarks:', BENCHMARK_DIR)
print('Checkpoint:', CHECKPOINT_DIR)
print('Results:', RESULTS_DIR)

In [ ]:
# Evaluation settings.
VAL_N = 4
MAX_TOKENS = 38912
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
GPU_MEMORY_UTILIZATION = 0.90
TENSOR_PARALLEL_SIZE = 1
MAX_MODEL_LEN = 40960
MAX_NUM_SEQS = 8
CHUNK_SIZE = 16
ENABLE_THINKING = True
LIMIT = None  # Set to 1 for smoke tests.

In [ ]:
# Imports and helpers.
import inspect
import json
import os
import re
import sys
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# vLLM V1 currently calls sys.stdout.fileno() during engine startup, which fails in Colab/ipykernel.
os.environ.setdefault('VLLM_USE_V1', '0')


class StreamWithFileno:
    def __init__(self, stream, fallback_fd: int):
        self._stream = stream
        self._fallback_fd = fallback_fd

    def fileno(self):
        try:
            return self._stream.fileno()
        except Exception:
            return self._fallback_fd

    def __getattr__(self, name):
        return getattr(self._stream, name)

    def write(self, data):
        return self._stream.write(data)

    def flush(self):
        return self._stream.flush()


def ensure_ipykernel_stream_fileno():
    for attr, fallback_fd in [('stdout', 1), ('stderr', 2)]:
        stream = getattr(sys, attr)
        try:
            stream.fileno()
        except Exception:
            setattr(sys, attr, StreamWithFileno(stream, fallback_fd))


ensure_ipykernel_stream_fileno()

import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest


def template_contains_name(template, name: str) -> bool:
    if isinstance(template, str):
        return name in template
    if isinstance(template, Mapping):
        return any(template_contains_name(value, name) for value in template.values())
    if isinstance(template, Sequence) and not isinstance(template, (str, bytes, bytearray)):
        return any(template_contains_name(value, name) for value in template)
    return False


def supports_chat_template_kwarg(tokenizer, name: str) -> bool:
    apply_chat_template = getattr(tokenizer, 'apply_chat_template')
    try:
        parameters = inspect.signature(apply_chat_template).parameters
    except (TypeError, ValueError):
        return template_contains_name(getattr(tokenizer, 'chat_template', None), name)
    if name in parameters:
        return True
    has_var_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values())
    return has_var_kwargs and template_contains_name(getattr(tokenizer, 'chat_template', None), name)


def apply_chat_template_compat(tokenizer, messages, *, enable_thinking=True):
    kwargs = {}
    if supports_chat_template_kwarg(tokenizer, 'enable_thinking'):
        kwargs['enable_thinking'] = enable_thinking
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        **kwargs,
    )


def build_prompt(tokenizer, problem: str) -> str:
    message = {
        'role': 'user',
        'content': f'{problem}\n\nPlease reason step by step, and put your final answer within \\boxed{{}}.',
    }
    return apply_chat_template_compat(tokenizer, [message], enable_thinking=ENABLE_THINKING)


def extract_boxed_answer(text: str | None) -> str | None:
    if not text:
        return None
    start = text.rfind('\\boxed')
    if start < 0:
        return None
    brace_start = text.find('{', start)
    if brace_start < 0:
        return None
    depth = 0
    for idx in range(brace_start, len(text)):
        if text[idx] == '{':
            depth += 1
        elif text[idx] == '}':
            depth -= 1
            if depth == 0:
                return text[brace_start + 1:idx].strip()
    return None


def grade_answer(predicted: str | None, ground_truth: str) -> bool:
    if predicted is None:
        return False
    try:
        from math_verify import parse, verify
        pred_expr = predicted if '$' in predicted else f'${predicted}$'
        gt_expr = ground_truth if '$' in ground_truth else f'${ground_truth}$'
        return bool(verify(parse(gt_expr, fallback_mode='no_fallback'), parse(pred_expr, fallback_mode='no_fallback'), timeout_seconds=5))
    except Exception:
        pred_norm = re.sub(r'[\s$,]', '', str(predicted)).lower()
        gt_norm = re.sub(r'[\s$,]', '', str(ground_truth)).lower()
        return pred_norm == gt_norm

In [ ]:
# Dataset loading with auto-detection from Drive.
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_table(path: Path) -> list[dict[str, Any]]:
    suffix = path.suffix.lower()
    if suffix == '.jsonl':
        return read_jsonl(path)
    if suffix == '.json':
        data = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(data, dict):
            for key in ('data', 'examples', 'rows'):
                if isinstance(data.get(key), list):
                    data = data[key]
                    break
        return list(data)
    if suffix == '.csv':
        return pd.read_csv(path).to_dict('records')
    if suffix == '.parquet':
        return pd.read_parquet(path).to_dict('records')
    raise ValueError(f'Unsupported benchmark file: {path}')


def find_benchmark_file(name: str) -> Path:
    candidates = []
    for ext in ('.jsonl', '.json', '.csv', '.parquet'):
        candidates.append(BENCHMARK_DIR / f'{name}{ext}')
    candidates.extend(sorted(BENCHMARK_DIR.glob(f'*{name}*')))
    for path in candidates:
        if path.exists() and path.is_file() and path.suffix.lower() in {'.jsonl', '.json', '.csv', '.parquet'}:
            return path
    raise FileNotFoundError(f'Could not find benchmark file for {name} under {BENCHMARK_DIR}')


def first_value(row: dict[str, Any], keys: tuple[str, ...], default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


def normalize_rows(name: str, rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    normalized = []
    for idx, row in enumerate(rows):
        problem = str(first_value(row, ('problem', 'question', 'prompt'), '')).strip()
        answer = str(first_value(row, ('answer', 'final_answer', 'target', 'ground_truth'), '')).strip()
        if not problem or not answer:
            continue
        normalized.append({
            'id': str(first_value(row, ('id', 'problem_id', 'problem_idx', 'question_id'), f'{name}_{idx}')),
            'source': str(first_value(row, ('source',), name)),
            'problem': problem,
            'answer': answer,
        })
    if LIMIT is not None:
        normalized = normalized[:LIMIT]
    return normalized

In [ ]:
# Load vLLM model with LoRA.
ensure_ipykernel_stream_fileno()
os.environ.setdefault('VLLM_USE_V1', '0')
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f'Missing LoRA checkpoint: {CHECKPOINT_DIR}')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
llm = LLM(
    model=BASE_MODEL,
    enable_lora=True,
    max_lora_rank=64,
    max_loras=1,
    max_cpu_loras=1,
    trust_remote_code=True,
    dtype='bfloat16',
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
)

sampling_params = SamplingParams(
    n=VAL_N,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)
lora_request = LoRARequest('qwen3_4b_avsd', 1, str(CHECKPOINT_DIR))
print('Loaded model and LoRA checkpoint')

In [ ]:
# Resume-aware evaluation.
def load_completed_records(path: Path) -> dict[str, dict[str, Any]]:
    if not path.exists():
        return {}
    completed = {}
    for row in read_jsonl(path):
        samples = row.get('samples') or []
        if len(samples) >= VAL_N:
            completed[str(row['id'])] = row
    return completed


def append_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def evaluate_benchmark(name: str) -> dict[str, Any]:
    input_path = find_benchmark_file(name)
    examples = normalize_rows(name, load_table(input_path))
    output_path = RESULTS_DIR / f'{name}_results.jsonl'
    summary_path = RESULTS_DIR / f'{name}_summary.json'
    completed = load_completed_records(output_path)
    pending = [ex for ex in examples if ex['id'] not in completed]

    print(f'{name}: {len(completed)} skipped, {len(pending)} pending, input={input_path}')
    for start in tqdm(range(0, len(pending), CHUNK_SIZE), desc=f'{name} generation chunks'):
        chunk = pending[start:start + CHUNK_SIZE]
        prompts = [build_prompt(tokenizer, ex['problem']) for ex in chunk]
        outputs = llm.generate(prompts, sampling_params=sampling_params, lora_request=lora_request, use_tqdm=False)
        new_records = []
        for ex, request_output in tqdm(list(zip(chunk, outputs)), desc=f'{name} grading', leave=False):
            samples = []
            for sample_idx, sample in enumerate(request_output.outputs):
                text = sample.text or tokenizer.decode(sample.token_ids, skip_special_tokens=False)
                predicted = extract_boxed_answer(text)
                correct = grade_answer(predicted, ex['answer'])
                samples.append({
                    'sample_index': sample_idx,
                    'predicted_answer': predicted,
                    'correct': correct,
                    'formatted': predicted is not None,
                    'generated_tokens': len(sample.token_ids),
                    'full_generation': text,
                })
            record = {
                'id': ex['id'],
                'source': ex['source'],
                'problem': ex['problem'],
                'ground_truth': ex['answer'],
                'samples': samples,
                'num_correct': sum(int(s['correct']) for s in samples),
            }
            completed[ex['id']] = record
            new_records.append(record)
        append_jsonl(output_path, new_records)

    all_records = list(completed.values())
    total_generations = sum(len(row.get('samples') or []) for row in all_records)
    correct_total = sum(int(sample.get('correct', False)) for row in all_records for sample in row.get('samples', []))
    formatted_total = sum(int(sample.get('formatted', False)) for row in all_records for sample in row.get('samples', []))
    summary = {
        'benchmark': name,
        'input_path': str(input_path),
        'output_path': str(output_path),
        'base_model': BASE_MODEL,
        'checkpoint_dir': str(CHECKPOINT_DIR),
        'val_n': VAL_N,
        'num_examples': len(all_records),
        'skipped_existing': len(examples) - len(pending),
        'evaluated_this_run': len(pending),
        'average_at_4_pct': 100.0 * correct_total / total_generations if total_generations else 0.0,
        'format_rate_pct': 100.0 * formatted_total / total_generations if total_generations else 0.0,
    }
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary

summaries = [evaluate_benchmark(name) for name in BENCHMARKS]
combined_path = RESULTS_DIR / 'combined_summary.json'
combined_path.write_text(json.dumps(summaries, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved combined summary to', combined_path)